
# Daily loss-ratio watch: detect, explain, recommend, notify

Runs every weekday morning after the overnight claims load.

1. **Detect**: read `v_loss_ratio_watchlist` and keep RED/AMBER segments.
2. **Explain**: ask the loss-ratio **Genie Agent** *why* each segment moved (frequency vs severity, causes, large losses).
   The question goes through the same governed agent that Genie One uses, so the explanation matches what an exec would get.
3. **Recommend**: map the driver to a deterministic **action playbook** per downstream team (Claims, Underwriting,
   Pricing, Reinsurance, Reserving, Finance). The playbook is rules, not an LLM, so actions are auditable.
4. **Record**: write alerts and actions to Delta (`lr_alerts`, `lr_action_recommendations`) for follow-up and audit.
   - 4b **Publish to Lakebase**: insert each action into the operational Postgres table `ops.lr_actions`
     (`ON CONFLICT DO NOTHING`, so a team's status, owner and note are never overwritten). Teams update it in the
     Action Desk app; `src/lakebase/03_snapshot_to_delta.py` copies it back as `lr_action_status`.
   - 4c **LLM judge (Gen AI)**: `ai_query` asks an independent model to grade every Genie explanation stored in
     `lr_alerts` (grounded in the numbers? names the driver? actionable?) and appends the verdicts to `lr_explanation_judge`,
     so explanation quality is tracked over time instead of assumed.
5. **Notify**: post a Slack message (Block Kit) and a Microsoft Teams card (Adaptive Card) to each team's channel.

In [ ]:
%pip install -q "databricks-sdk>=0.65" "psycopg[binary]>=3.2"

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [ ]:
dbutils.library.restartPython()

In [ ]:
dbutils.widgets.text("catalog", "")
dbutils.widgets.text("schema", "loss_ratio_coworker")
dbutils.widgets.text("genie_space_id", "")
dbutils.widgets.text("secret_scope", "loss_ratio_coworker")
dbutils.widgets.dropdown("dry_run", "true", ["true", "false"])
dbutils.widgets.text("lakebase_instance", "lrc-lakebase")
dbutils.widgets.text("lakebase_database", "lrc_ops")
dbutils.widgets.text("judge_endpoint", "databricks-claude-sonnet-5-5")

CATALOG = dbutils.widgets.get("catalog") or spark.sql("SELECT current_catalog()").first()[0]
SCHEMA = dbutils.widgets.get("schema")
SPACE_ID = dbutils.widgets.get("genie_space_id")
SCOPE = dbutils.widgets.get("secret_scope")
DRY_RUN = dbutils.widgets.get("dry_run") == "true"
LAKEBASE_INSTANCE = dbutils.widgets.get("lakebase_instance")
LAKEBASE_DB = dbutils.widgets.get("lakebase_database")
JUDGE = dbutils.widgets.get("judge_endpoint")
FQ = f"`{CATALOG}`.`{SCHEMA}`"

## 1 · Detect

In [ ]:
watch = spark.sql(f"""
  SELECT * FROM {FQ}.v_loss_ratio_watchlist
  WHERE status IN ('RED','AMBER')
  ORDER BY CASE status WHEN 'RED' THEN 0 ELSE 1 END, gap_7d_pts DESC
""").toPandas()
benchmark = spark.sql(f"""
  SELECT value FROM {FQ}.industry_benchmarks
  WHERE benchmark = 'US P&C industry' AND period = 'FY2025'
""").first()[0]
book = spark.sql(f"""
  SELECT MEASURE(loss_ratio) lr, MEASURE(combined_ratio) cr
  FROM {FQ}.mv_loss_ratio WHERE YEAR(as_of_date) = YEAR(current_date())
""").first()
print(f"{len(watch)} segments need attention | book YTD LR {book.lr:.1%}, COR {book.cr:.1%} vs industry {benchmark:.1%}")
display(watch)

6 segments need attention | book YTD LR 63.6%, COR 92.1% vs industry 92.9%


| line_of_business | region | owner_team | as_of_date | target_loss_ratio | amber_pts | red_pts | lr_7d | lr_30d | lr_ytd | freq_7d | freq_base | sev_7d | sev_base | large_losses_7d | cat_share_7d | incurred_7d | gap_7d_pts | gap_30d_pts | freq_change | sev_change | primary_driver | status |
|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|
| Property | DACH | Property Underwriting | 2026-11-16 | 0.6 | 10.0 | 25.0 | 2.6593116321170864 | 0.9810150961485058 | 0.6589842631991109 | 61.18764803764787 | 9.87173589452217 | 43461.576272401435 | 58994.76312080538 | 0 | 0.8310380810824852 | 12125779.78 | 205.9 | 38.1 | 5.198 | -0.263 | Catastrophe event | RED |
| Liability | Southern Europe | Casualty Underwriting | 2026-11-16 | 0.66 | 10.0 | 25.0 | 2.0897577713813615 | 0.9401509447041853 | 0.6953967964638149 | 9.804896189474686 | 9.427890889819096 | 213134.1047368421 | 68337.16806584361 | 3 | 0.0 | 4049547.9899999998 | 143.0 | 28.0 | 0.04 | 2.119 | Large loss | RED |
| Motor | UK & Ireland | Motor Underwriting | 2026-11-16 | 0.7 | 8.0 | 20.0 | 1.0351407816835614 | 0.8408605666612735 | 0.7166413021377629 | 50.09747518480893 | 38.20985347860525 | 20662.533947368418 | 19236.146901041666 | 0 | 0.0 | 3140705.1599999997 | 33.5 | 14.1 | 0.311 | 0.074 | Frequency | RED |
| Liability | UK & Ireland | Casualty Underwriting | 2026-11-16 | 0.66 | 10.0 | 25.0 | 0.7945636047723654 | 0.6662629069861268 | 0.6989768635385595 | 11.031849800895216 | 9.846168375226322 | 72024.51258064517 | 74662.4911764706 | 1 | 0.0 | 2232759.89 | 13.5 | 0.6 | 0.12 | -0.035 | Within normal variation | AMBER |
| Property | UK & Ireland | Property Underwriting | 2026-11-16 | 0.6 | 10.0 | 25.0 | 0.7282354841240234 | 0.5929742840625615 | 0.5901111262696466 | 11.489493093775177 | 9.961729376821383 | 63382.73396226416 | 57064.144311183154 | 1 | 0.0 | 3359284.9000000004 | 12.8 | -0.7 | 0.153 | 0.111 | Within normal variation | AMBER |
| Liability | France | Casualty Underwriting | 2026-11-16 | 0.66 | 10.0 | 25.0 | 0.7677543674222962 | 0.5835748427629629 | 0.6320859784372923 | 13.43314073743711 | 10.698533409745693 | 57153.75 | 62068.2673421927 | 0 | 0.0 | 1600305.0 | 10.8 | -7.6 | 0.256 | -0.079 | Frequency | AMBER |

## 2 · Explain with the Genie Agent

In [ ]:
import time
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()

def ask_genie(question: str) -> str:
    """Genie Conversation API over REST (works with any databricks-sdk version on serverless)."""
    if not SPACE_ID:
        return ""
    try:
        base = f"/api/2.0/genie/spaces/{SPACE_ID}"
        start = w.api_client.do("POST", f"{base}/start-conversation", body={"content": question})
        cid, mid = start["conversation_id"], start["message_id"]
        for _ in range(60):
            msg = w.api_client.do("GET", f"{base}/conversations/{cid}/messages/{mid}")
            if msg.get("status") in ("COMPLETED", "FAILED", "CANCELLED"):
                break
            time.sleep(3)
        atts = msg.get("attachments") or []
        parts = [a["text"]["content"] for a in atts if a.get("text", {}).get("content")]
        if not parts:
            parts = [a["query"]["description"] for a in atts if a.get("query", {}).get("description")]
        return "\n".join(parts).strip()
    except Exception as e:  # never block the alert on the explanation
        return f"(Genie explanation unavailable: {e})"

explanations = {}
for r in watch.itertuples():
    q = (f"Why did the {r.line_of_business} loss ratio in {r.region} rise over the last 7 days? "
         f"Split it into claim frequency versus severity, list the top causes of loss and any large or "
         f"catastrophe losses, and compare it with the target loss ratio. Answer in three short bullet points.")
    explanations[(r.line_of_business, r.region)] = ask_genie(q)

## 3 · Recommend: the action playbook

In [ ]:
PLAYBOOK = {
    "Catastrophe event": [
        ("Claims", "Stand up the event response: surge adjusters, fast-track triage of the event claims, daily event count.", "24h"),
        ("Reinsurance", "Notify cat XL reinsurers of a potential event and estimate the recovery against the retention.", "48h"),
        ("Reserving", "Book an event IBNR estimate so the month-end loss ratio reflects claims not yet reported.", "5 days"),
        ("Underwriting", "Check accumulation in the affected area and hold new binds above the referral limit.", "48h"),
        ("Finance", "Flag the event in the month-end outlook with gross and net-of-reinsurance impact.", "Month-end"),
    ],
    "Frequency": [
        ("Underwriting", "Review risk selection and rate adequacy on renewals in this segment; check broker and channel mix.", "Next renewal cycle"),
        ("Pricing", "Test whether the frequency trend is in the rating model and price a frequency load if it persists.", "2 weeks"),
        ("Claims", "Screen the cluster of new claims for fraud indicators and repeat insureds.", "1 week"),
    ],
    "Large loss": [
        ("Claims", "Assign a senior handler, convene the large-loss committee and review the case reserve.", "48h"),
        ("Reinsurance", "Check whether the claim reaches the per-risk XL attachment and notify reinsurers.", "1 week"),
        ("Underwriting", "Review the account and similar risks before renewal.", "Before renewal"),
    ],
    "Severity": [
        ("Claims", "Analyse repair, supplier and legal cost inflation on the affected causes of loss.", "2 weeks"),
        ("Pricing", "Refresh the severity trend assumption in pricing.", "Next pricing review"),
    ],
    "Within normal variation": [
        ("Underwriting", "Monitor. No action unless the 30-day loss ratio also breaches.", "Next daily run"),
    ],
}

alerts, actions = [], []
for r in watch.itertuples():
    alert_id = f"LR-{r.as_of_date:%Y%m%d}-{r.line_of_business[:3].upper()}-{r.region.split()[0][:4].upper()}"
    alerts.append(dict(alert_id=alert_id, as_of_date=r.as_of_date, line_of_business=r.line_of_business,
                       region=r.region, status=r.status, primary_driver=r.primary_driver,
                       lr_7d=float(r.lr_7d), lr_30d=float(r.lr_30d), target_loss_ratio=float(r.target_loss_ratio),
                       gap_7d_pts=float(r.gap_7d_pts), freq_change=float(r.freq_change or 0),
                       sev_change=float(r.sev_change or 0), large_losses_7d=int(r.large_losses_7d),
                       genie_explanation=explanations[(r.line_of_business, r.region)]))
    for team, action, due in PLAYBOOK[r.primary_driver]:
        actions.append(dict(alert_id=alert_id, team=team, action=action, due_within=due,
                            priority="P1" if r.status == "RED" else "P2", state="Open"))

## 4 · Record (append-only audit trail)

In [ ]:
import pandas as pd
from pyspark.sql import functions as F

if alerts:
    (spark.createDataFrame(pd.DataFrame(alerts)).withColumn("created_at", F.current_timestamp())
        .write.mode("append").option("mergeSchema", "true").saveAsTable(f"{FQ}.lr_alerts"))
    (spark.createDataFrame(pd.DataFrame(actions)).withColumn("created_at", F.current_timestamp())
        .write.mode("append").option("mergeSchema", "true").saveAsTable(f"{FQ}.lr_action_recommendations"))
print(f"recorded {len(alerts)} alerts and {len(actions)} actions")

recorded 6 alerts and 16 actions


## 4b · Publish the actions to Lakebase (operational store for team status)
Delta keeps the audit trail; Lakebase Postgres holds the rows teams change during the day. The insert is idempotent and
never overwrites a team's update.

In [ ]:
%run ../lakebase/_lakebase_conn

In [ ]:
inst = ensure_instance(LAKEBASE_INSTANCE)
with connect(inst, LAKEBASE_DB) as c:
    with c.transaction():
        new = 0
        for x in actions:
            a = next(al for al in alerts if al["alert_id"] == x["alert_id"])
            cur = c.execute("""INSERT INTO ops.lr_actions (action_id, alert_id, as_of_date, line_of_business, region, alert_status,
                                 primary_driver, team, priority, action, due_within)
                               VALUES (%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s) ON CONFLICT (action_id) DO NOTHING""",
                            (f"{x['alert_id']}:{x['team']}", x["alert_id"], a["as_of_date"], a["line_of_business"], a["region"],
                             a["status"], a["primary_driver"], x["team"], x["priority"], x["action"], x["due_within"]))
            new += cur.rowcount
    status = c.execute("SELECT team, status, count(*) FROM ops.lr_actions GROUP BY team, status ORDER BY team, status").fetchall()
print(f"Lakebase ops.lr_actions: {new} new actions published, {len(actions) - new} already there (team updates kept)")
for team, st, n in status:
    print(f"  {team:<12} {st:<12} {n}")

Lakebase ops.lr_actions: 0 new actions published, 16 already there (team updates kept)
  Claims       ACKNOWLEDGED 1
  Claims       IN_PROGRESS  1
  Claims       OPEN         2
  Finance      OPEN         1
  Pricing      OPEN         2
  Reinsurance  OPEN         2
  Reserving    OPEN         1
  Underwriting OPEN         6


## 4c · LLM judge: grade today's Genie explanations (Gen AI quality, tracked over time)

In [ ]:
judged = spark.sql(f"""
WITH today AS (
  SELECT alert_id, line_of_business, region, status, primary_driver, lr_7d, target_loss_ratio, genie_explanation
  FROM {FQ}.lr_alerts WHERE as_of_date = (SELECT MAX(as_of_date) FROM {FQ}.lr_alerts) AND genie_explanation IS NOT NULL
),
j AS (
  SELECT *, ai_query('{JUDGE}', concat(
    'You audit explanations written by an analytics assistant for insurance executives. Reply with JSON only: ',
    '{{"grounded": true|false, "names_driver": true|false, "actionable": true|false, "score": 1-5, "reason": "<max 20 words>"}}. ',
    'grounded: its numbers agree with the FACTS. names_driver: it identifies the driver given in the FACTS. ',
    'actionable: an executive knows what to look at next. ',
    'FACTS: segment ', line_of_business, ' / ', region, ', status ', status, ', 7-day loss ratio ', cast(round(lr_7d * 100, 1) AS STRING),
    '%, target ', cast(round(target_loss_ratio * 100, 1) AS STRING), '%, driver ', primary_driver,
    '. EXPLANATION: ', genie_explanation)) AS verdict_raw
  FROM today
)
SELECT current_timestamp() AS judged_at, '{JUDGE}' AS judge_endpoint, alert_id, line_of_business, region, primary_driver,
       from_json(regexp_extract(verdict_raw, '[{{].*[}}]', 0),
                 'grounded BOOLEAN, names_driver BOOLEAN, actionable BOOLEAN, score INT, reason STRING') AS v
FROM j
""").selectExpr("judged_at", "judge_endpoint", "alert_id", "line_of_business", "region", "primary_driver", "v.*")
judged.write.mode("append").option("mergeSchema", "true").saveAsTable(f"{FQ}.lr_explanation_judge")
spark.sql(f"COMMENT ON TABLE {FQ}.lr_explanation_judge IS 'LLM-judge verdicts on the Genie explanations stored in lr_alerts, appended every run.'")
display(judged.select("alert_id", "primary_driver", "grounded", "names_driver", "actionable", "score", "reason"))
display(spark.sql(f"""SELECT date(judged_at) AS day, COUNT(*) AS explanations, ROUND(AVG(score), 2) AS avg_score,
                             ROUND(AVG(CAST(grounded AS INT)), 2) AS grounded, ROUND(AVG(CAST(names_driver AS INT)), 2) AS names_driver
                      FROM {FQ}.lr_explanation_judge GROUP BY ALL ORDER BY day"""))

| alert_id | primary_driver | grounded | names_driver | actionable | score | reason |
|---|---|---|---|---|---|---|
| LR-20261116-PRO-DACH | Catastrophe event | True | True | True | 4 | Figures match the FACTS and the 205.9-point gap is correct; '+5.198x' is ambiguous (the ratio is about 6.2x), but the hail event is clearly named. |
| LR-20261116-LIA-SOUT | Large loss | True | True | True | 4 | Figures match the FACTS and large loss is named. The 4.0% frequency rise is slightly off (about 3.9%). The next step is implied rather than stated. |
| LR-20261116-MOT-UK | Frequency | True | True | True | 4 | Loss ratio, target, gap and change percentages check out; frequency named as driver; collision focus points to next steps, though no explicit recommendation. |
| LR-20261116-LIA-UK | Within normal variation | True | False | False | 2 | Headline numbers match, but it asserts a frequency-led driver instead of 'Within normal variation' and gives no next step. |
| LR-20261116-PRO-UK | Within normal variation | True | True | True | 4 | 72.8%, 60.0% and the 12.8-point gap match the FACTS, and the driver is cited. Other figures are unverifiable. It points to Water Damage and frequency but gives no explicit next step. |
| LR-20261116-LIA-FRAN | Frequency | True | True | True | 4 | Figures reconcile with the facts and internally; frequency is named as the driver. No explicit next step, but cause breakdown guides review. |
| LR-20261116-PRO-DACH | Catastrophe event | True | True | True | 4 | Loss ratio, target and 205.9-point gap match the FACTS; catastrophe driver named; Hailstorm DACH gives a clear focus, though the frequency/severity units are unclear. |
| LR-20261116-LIA-SOUT | Large loss | True | True | True | 4 | Loss ratio, target and gap match the FACTS, and Large loss is named. Extra figures are consistent but unverifiable; no explicit next step. |
| LR-20261116-MOT-UK | Frequency | True | True | True | 3 | Loss ratio, target, gap and Frequency driver match the FACTS. Duplicate Collision rows and unexplained cat incurred amount alongside 'Attritional' tags confuse the picture. |
| LR-20261116-LIA-UK | Within normal variation | True | True | False | 2 | Headline figures match the FACTS, but the frequency-over-severity claim conflicts with its own severity data. Raw unrounded numbers and no clear next step. |
| LR-20261116-PRO-UK | Within normal variation | True | True | True | 3 | Stated figures match the FACTS and the driver is named, but 'driven by frequency and severity' conflicts with 'normal variation'. Extra figures can't be verified, and no next step is given. |
| LR-20261116-LIA-FRAN | Frequency | True | True | True | 4 | Loss ratio, target, and 10.8-point gap match FACTS; frequency named as driver; extra figures unverifiable; top causes guide next steps. |
| LR-20261116-PRO-DACH | Catastrophe event | True | True | True | 4 | Figures are internally consistent and match the facts. Catastrophe losses and the Hailstorm DACH event are named, though only as a secondary point after frequency. |
| LR-20261116-LIA-SOUT | Large loss | True | True | True | 4 | Figures are consistent with the FACTS and each other; severity and 3 large losses (EUR 3.14m) are the driver, though not explicitly labeled. |
| LR-20261116-MOT-UK | Frequency | True | True | False | 3 | Figures match the FACTS, but it dilutes the frequency driver with severity, shows catastrophe EUR 3.14M above the 'top' Attritional EUR 2.25M, and gives no clear next step. |
| LR-20261116-LIA-UK | Within normal variation | True | False | True | 3 | Numbers are consistent with the FACTS, but it never says the driver is within normal variation and instead blames severity. |
| LR-20261116-PRO-UK | Within normal variation | True | False | True | 3 | 72.8%, 60.0% target and 12.8-point gap match; but it never says 'within normal variation', instead blaming frequency and severity for a 34-point rise. |
| LR-20261116-LIA-FRAN | Frequency | True | True | True | 3 | Figures match FACTS and frequency is named as driver; but the opening question is off-format and calling all claims catastrophe/event is dubious. |
| LR-20261116-PRO-DACH | Catastrophe event | True | True | True | 4 | Loss ratio, target, gap (205.9 pts) and driver match the FACTS; extra figures are unverifiable, and 'Attritional' as catastrophe-related is muddled. |
| LR-20261116-LIA-SOUT | Large loss | True | True | True | 4 | Loss ratio, target, 143-point gap and large-loss driver match FACTS; extra figures are unverifiable but plausible; points to specific large claims. |
| LR-20261116-MOT-UK | Frequency | True | True | True | 4 | Gap of 33.5 points is correct and Frequency is named. Collision and frequency creep give direction, but no explicit next step is recommended. |
| LR-20261116-LIA-UK | Within normal variation | True | True | True | 3 | Facts and arithmetic check out, but it also attributes the move to frequency despite the 'normal variation' driver, and offers no explicit next step. |
| LR-20261116-PRO-UK | Within normal variation | True | True | True | 3 | Facts match and driver is named, but the 3M large loss exceeds the top cause total, and no explicit next step is given. |
| LR-20261116-LIA-FRAN | Frequency | True | True | True | 4 | Figures reconcile (gap 10.8pp, severity -7.9%, 28×57,153.75=1,600,305); frequency named as driver; the +25.6% is a minor rounding difference, and no explicit next step is given. |
| LR-20261116-PRO-DACH | Catastrophe event | True | True | True | 4 | Figures match FACTS and are internally consistent; the catastrophe driver appears via Hailstorm DACH and cat-tagged losses, though the headline blames frequency. |
| LR-20261116-LIA-SOUT | Large loss | True | True | True | 4 | 209.0%, 66.0% and the 143.0pt gap match the FACTS; severity and 3 large losses (EUR 3.1m) point to the large-loss driver, though it is not named explicitly. |
| LR-20261116-MOT-UK | Frequency | False | False | False | 1 | Restates the question as a request; gives no figures (103.5%, 70.0%) and never identifies Frequency as the driver. |
| LR-20261116-LIA-UK | Within normal variation | True | False | False | 2 | Figures are consistent with the FACTS, but it never says 'within normal variation' and instead asserts frequency and severity drove the rise, misleading readers. |
| LR-20261116-PRO-UK | Within normal variation | False | False | False | 2 | Loss ratio and target gap match, but it claims a frequency/severity-driven rise, contradicting 'Within normal variation'; other figures unsupported. |
| LR-20261116-LIA-FRAN | Frequency | True | True | True | 4 | Gap and delta arithmetic match the FACTS and internal figures reconcile. Frequency is named, and the top loss causes point to where to look next. No explicit recommendation. |

_6 more rows in the CSV._

| day | explanations | avg_score | grounded | names_driver |
|---|---|---|---|---|
| 2026-11-16 | 36 | 3.14 | 0.81 | 0.72 |

## 5 · Notify Slack and Microsoft Teams
Webhook URLs live in a secret scope (`slack_webhook_url`, `teams_webhook_url`). With `dry_run=true` the payloads are
printed instead of sent.

In [ ]:
import json, requests

ICON = {"RED": ":red_circle:", "AMBER": ":large_orange_circle:"}

def slack_payload(a, acts):
    lines = "\n".join(f"• *{x['team']}* ({x['due_within']}): {x['action']}" for x in acts)
    return {"text": f"Loss ratio {a['status']}: {a['line_of_business']} / {a['region']}", "blocks": [
        {"type": "header", "text": {"type": "plain_text", "text": f"Loss ratio alert: {a['line_of_business']}, {a['region']}"}},
        {"type": "section", "fields": [
            {"type": "mrkdwn", "text": f"*Status*\n{ICON[a['status']]} {a['status']}"},
            {"type": "mrkdwn", "text": f"*Driver*\n{a['primary_driver']}"},
            {"type": "mrkdwn", "text": f"*7-day LR*\n{a['lr_7d']:.0%} vs target {a['target_loss_ratio']:.0%} (+{a['gap_7d_pts']:.0f} pts)"},
            {"type": "mrkdwn", "text": f"*30-day LR*\n{a['lr_30d']:.0%}"}]},
        {"type": "section", "text": {"type": "mrkdwn", "text": f"*Why (Genie)*\n{a['genie_explanation'][:2500] or '_n/a_'}"}},
        {"type": "section", "text": {"type": "mrkdwn", "text": f"*Recommended actions*\n{lines}"}},
        {"type": "context", "elements": [{"type": "mrkdwn", "text":
            f"Alert {a['alert_id']} · book YTD COR {book.cr:.1%} vs US P&C FY2025 {benchmark:.1%} (AM Best) · ask follow-ups with @Genie"}]},
    ]}

def teams_payload(a, acts):
    return {"type": "message", "attachments": [{"contentType": "application/vnd.microsoft.card.adaptive", "content": {
        "$schema": "http://adaptivecards.io/schemas/adaptive-card.json", "type": "AdaptiveCard", "version": "1.4",
        "body": [
            {"type": "TextBlock", "size": "Large", "weight": "Bolder",
             "text": f"Loss ratio {a['status']}: {a['line_of_business']}, {a['region']}",
             "color": "Attention" if a["status"] == "RED" else "Warning"},
            {"type": "FactSet", "facts": [
                {"title": "Driver", "value": a["primary_driver"]},
                {"title": "7-day LR", "value": f"{a['lr_7d']:.0%} (target {a['target_loss_ratio']:.0%}, +{a['gap_7d_pts']:.0f} pts)"},
                {"title": "30-day LR", "value": f"{a['lr_30d']:.0%}"},
                {"title": "Alert", "value": a["alert_id"]}]},
            {"type": "TextBlock", "wrap": True, "text": f"**Why (Genie)** {a['genie_explanation'][:1500]}"},
            {"type": "TextBlock", "weight": "Bolder", "text": "Recommended actions"},
            *[{"type": "TextBlock", "wrap": True, "text": f"- **{x['team']}** ({x['due_within']}): {x['action']}"} for x in acts],
        ]}}]}

def secret(key):
    try:
        return dbutils.secrets.get(SCOPE, key)
    except Exception:
        return None

slack_url, teams_url = secret("slack_webhook_url"), secret("teams_webhook_url")
for a in alerts:
    acts = [x for x in actions if x["alert_id"] == a["alert_id"]]
    for name, url, payload in [("slack", slack_url, slack_payload(a, acts)), ("teams", teams_url, teams_payload(a, acts))]:
        if DRY_RUN or not url:
            print(f"[{name} dry-run] {a['alert_id']}\n{json.dumps(payload)[:600]}...\n")
        else:
            resp = requests.post(url, json=payload, timeout=20)
            print(f"[{name}] {a['alert_id']} -> HTTP {resp.status_code}")

[slack dry-run] LR-20261116-PRO-DACH
{"text": "Loss ratio RED: Property / DACH", "blocks": [{"type": "header", "text": {"type": "plain_text", "text": "Loss ratio alert: Property, DACH"}}, {"type": "section", "fields": [{"type": "mrkdwn", "text": "*Status*\n:red_circle: RED"}, {"type": "mrkdwn", "text": "*Driver*\nCatastrophe event"}, {"type": "mrkdwn", "text": "*7-day LR*\n266% vs target 60% (+206 pts)"}, {"type": "mrkdwn", "text": "*30-day LR*\n98%"}]}, {"type": "section", "text": {"type": "mrkdwn", "text": "*Why (Genie)*\n- **Property DACH 7-day loss ratio is 265.9%**, up from **46.3%** in the prior 7 days and **205.9 percentag...

[teams dry-run] LR-20261116-PRO-DACH
{"type": "message", "attachments": [{"contentType": "application/vnd.microsoft.card.adaptive", "content": {"$schema": "http://adaptivecards.io/schemas/adaptive-card.json", "type": "AdaptiveCard", "version": "1.4", "body": [{"type": "TextBlock", "size": "Large", "weight": "Bolder", "text": "Loss ratio RED: Property, DACH